# Customer Segmentation Using K-Means Clustering

**Objective:** Group mall customers according to annual income and spending score to identify useful customer segments.

**Algorithm:** K-Means clustering  
**Dataset:** Mall Customers (`Mall_Customers.csv`)  
**Features used:** Annual Income (k$), Spending Score (1-100)

> Keep `Mall_Customers.csv` in the same folder as this notebook before running the cells.

## 1. Import required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

## 2. Load the dataset

In [ ]:
df = pd.read_csv("Mall_Customers.csv")
print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

## 3. Explore and check the data

In [ ]:
print("Dataset information:")
df.info()

print("\\nMissing values in each column:")
display(df.isnull().sum())

print("\\nSummary statistics:")
display(df.describe(include="all"))

In [ ]:
# Remove duplicate rows, if any
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)

# Check that the required columns exist
features = ['Annual Income (k$)', 'Spending Score (1-100)']
missing_columns = [col for col in features if col not in df.columns]

if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

# Handle missing values in the selected features
df = df.dropna(subset=features).copy()
print("Rows available for clustering:", len(df))

## 4. Select features and visualize the original data

In [ ]:
X = df[features].copy()

plt.figure(figsize=(8, 6))
plt.scatter(X[features[0]], X[features[1]], alpha=0.75)
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.title("Customer Distribution Before Clustering")
plt.grid(alpha=0.2)
plt.show()

## 5. Standardize the features

Standardization puts both features on a comparable scale before calculating distances. The scaler is fitted only on the selected clustering features.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 6. Find a suitable number of clusters (Elbow Method)

In [ ]:
wcss = []
k_values = range(1, 11)

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), wcss, marker='o')
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Within-Cluster Sum of Squares (WCSS)")
plt.title("Elbow Method")
plt.xticks(list(k_values))
plt.grid(alpha=0.25)
plt.show()

## 7. Compare cluster quality using Silhouette Score

The Silhouette Score ranges from -1 to 1. Higher values generally indicate better-separated clusters. Use it together with the Elbow graph and interpretability when choosing `best_k`.

In [ ]:
silhouette_scores = {}

for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    silhouette_scores[k] = silhouette_score(X_scaled, labels)

for k, score in silhouette_scores.items():
    print(f"k={k}: silhouette score={score:.3f}")

plt.figure(figsize=(8, 5))
plt.plot(list(silhouette_scores.keys()),
         list(silhouette_scores.values()), marker='o')
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different Cluster Counts")
plt.xticks(list(silhouette_scores.keys()))
plt.grid(alpha=0.25)
plt.show()

## 8. Train the final K-Means model

The original notebook used five clusters. We retain `k=5` as the standard Mall Customers demonstration choice; check the two evaluation plots before finalizing this choice for your dataset.

In [ ]:
best_k = 5

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

final_silhouette = silhouette_score(X_scaled, df['Cluster'])
print(f"Number of clusters: {best_k}")
print(f"Final silhouette score: {final_silhouette:.3f}")

display(df.head())

## 9. Visualize the customer segments

In [ ]:
# Convert the cluster centres back to the original feature scale for plotting
centres_original = scaler.inverse_transform(kmeans.cluster_centers_)

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df,
    x=features[0],
    y=features[1],
    hue='Cluster',
    palette='tab10',
    s=75,
    alpha=0.85
)
plt.scatter(
    centres_original[:, 0],
    centres_original[:, 1],
    s=230,
    c='black',
    marker='X',
    label='Cluster centres'
)
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.title("Customer Segments Using K-Means")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 10. Analyze the customer groups

In [ ]:
cluster_profile = df.groupby('Cluster').agg(
    Customer_Count=('Cluster', 'size'),
    Average_Annual_Income=('Annual Income (k$)', 'mean'),
    Average_Spending_Score=('Spending Score (1-100)', 'mean')
).round(2)

display(cluster_profile)
print("Number of customers in each cluster:")
display(df['Cluster'].value_counts().sort_index())

### Interpreting the clusters

Use the cluster profile above to describe each group based on its average income and spending score. For example, a cluster with high average income and high average spending score can be described as a high-income, high-spending group. Do not assign descriptions without checking the actual values in your output.

## 11. Save the results and trained objects

In [ ]:
df.to_csv("Customer_Segments.csv", index=False)
joblib.dump(kmeans, "kmeans_customer_model.pkl")
joblib.dump(scaler, "customer_feature_scaler.pkl")

print("Saved: Customer_Segments.csv")
print("Saved: kmeans_customer_model.pkl")
print("Saved: customer_feature_scaler.pkl")

## 12. Conclusion

K-Means clustering was applied to group customers using annual income and spending score. The Elbow Method and Silhouette Score were used to help assess the clustering structure. The cluster profile and scatter plot make the resulting customer groups easier to interpret.

**Limitation:** The results depend on the selected features, dataset quality, scaling method and chosen number of clusters. The groups describe patterns in this dataset and should not be treated as definitive customer labels.